In [18]:
import pandas as pd

In [19]:
data = pd.read_csv("../data/loan_approval_dataset.csv")
data.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [20]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   loan_id                    4269 non-null   int64
 1    no_of_dependents          4269 non-null   int64
 2    education                 4269 non-null   str  
 3    self_employed             4269 non-null   str  
 4    income_annum              4269 non-null   int64
 5    loan_amount               4269 non-null   int64
 6    loan_term                 4269 non-null   int64
 7    cibil_score               4269 non-null   int64
 8    residential_assets_value  4269 non-null   int64
 9    commercial_assets_value   4269 non-null   int64
 10   luxury_assets_value       4269 non-null   int64
 11   bank_asset_value          4269 non-null   int64
 12   loan_status               4269 non-null   str  
dtypes: int64(10), str(3)
memory usage: 433.7 KB


In [22]:
data[" loan_status"] = data[" loan_status"].map({" Approved":1," Rejected":0})

In [23]:
x = data.drop(columns=["loan_id"," loan_status"])
y = data[" loan_status"]
print(x)
print(y)

      no_of_dependents      education self_employed  income_annum  \
0                    2       Graduate            No       9600000   
1                    0   Not Graduate           Yes       4100000   
2                    3       Graduate            No       9100000   
3                    3       Graduate            No       8200000   
4                    5   Not Graduate           Yes       9800000   
...                ...            ...           ...           ...   
4264                 5       Graduate           Yes       1000000   
4265                 0   Not Graduate           Yes       3300000   
4266                 2   Not Graduate            No       6500000   
4267                 1   Not Graduate            No       4100000   
4268                 1       Graduate            No       9200000   

      loan_amount  loan_term  cibil_score  residential_assets_value  \
0        29900000         12          778                   2400000   
1        12200000          8 

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

num_features = x.select_dtypes(include=["int64"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline,num_features),
    ("cat",cat_pipeline,cat_features)
])

C:\Users\nigam\AppData\Local\Temp\ipykernel_2160\3173069230.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = x.select_dtypes(include=["object"]).columns


In [26]:
x_preporcessed = preprocessor.fit_transform(x)
x_preporcessed

array([[-0.2941017 ,  1.61797904,  1.63305171, ...,  0.        ,
         1.        ,  0.        ],
       [-1.4735476 , -0.34174956, -0.32441406, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.29562125,  1.4398219 ,  1.61093345, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.2941017 ,  0.51340474,  0.96950399, ...,  1.        ,
         1.        ,  0.        ],
       [-0.88382465, -0.34174956, -0.25805929, ...,  1.        ,
         1.        ,  0.        ],
       [-0.88382465,  1.47545332,  1.61093345, ...,  0.        ,
         1.        ,  0.        ]], shape=(4269, 13))

In [28]:
import torch

x_tensor = torch.tensor(x_preporcessed,dtype=torch.float32)
y_tensor = torch.tensor(y,dtype=torch.float32).view(-1,1)
print(x_tensor)
print(y_tensor)

tensor([[-0.2941,  1.6180,  1.6331,  ...,  0.0000,  1.0000,  0.0000],
        [-1.4735, -0.3417, -0.3244,  ...,  1.0000,  0.0000,  1.0000],
        [ 0.2956,  1.4398,  1.6109,  ...,  0.0000,  1.0000,  0.0000],
        ...,
        [-0.2941,  0.5134,  0.9695,  ...,  1.0000,  1.0000,  0.0000],
        [-0.8838, -0.3417, -0.2581,  ...,  1.0000,  1.0000,  0.0000],
        [-0.8838,  1.4755,  1.6109,  ...,  0.0000,  1.0000,  0.0000]])
tensor([[1.],
        [0.],
        [0.],
        ...,
        [0.],
        [1.],
        [1.]])


In [29]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self,input_size):
        super(MLP,self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size,64),
            nn.ReLU(),
            
            nn.Linear(64,32),
            nn.ReLU(),
            
            nn.Linear(32,1),
            nn.Sigmoid()
            
        )
    
    def forward(self,x):
        return self.model(x)    

In [30]:
model = MLP(input_size=x_tensor.shape[1])

In [31]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [34]:
epochs = 1001

for epoch in range(epochs):
    
    outputs = model(x_tensor)
    
    loss = criterion(outputs,y_tensor)
    
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch%100 == 0:
        print(f"Epoch : {epoch} ,Loss : {loss.item():.4f}")
        

Epoch : 0 ,Loss : 0.0212
Epoch : 100 ,Loss : 0.0164
Epoch : 200 ,Loss : 0.0128
Epoch : 300 ,Loss : 0.0101
Epoch : 400 ,Loss : 0.0082
Epoch : 500 ,Loss : 0.0066
Epoch : 600 ,Loss : 0.0055
Epoch : 700 ,Loss : 0.0045
Epoch : 800 ,Loss : 0.0038
Epoch : 900 ,Loss : 0.0033
Epoch : 1000 ,Loss : 0.0028


In [35]:
with torch.no_grad():
    preds = model(x_tensor)
    predicted = (preds > 0.5).float()
    
print(predicted[:10])    

tensor([[1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.]])


In [36]:
accuracy = (predicted == y_tensor).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.9997657537460327
